# Avaliacao externa comparativa da Fase 3.6 no Mirage-Test 2000 v1

As 2.000 imagens ja foram congeladas e usadas na avaliacao da 3.5. Este notebook reutiliza as mesmas imagens e o mesmo protocolo, com pesos, temperatura e limiar da 3.6 fixados no treino interno. O Mirage ja foi observado: esta e uma comparacao externa, nao um teste cego independente para desenvolver outra fase. Nada inicia automaticamente. Execute as celulas 1-5, depois 6 (copia local), 7 (congelamento) e 8 (avaliacao) manualmente. A celula 9 apenas le os resultados.


In [1]:
# 1. GPU e dependências do runtime.
import torch, sys, subprocess
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab antes de continuar'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cryptography', 'scikit-learn'])
print('Dependências disponíveis.', flush=True)

GPU: Tesla T4
Dependências disponíveis.


In [2]:
# 2. Drive e pastas novas; não altera experimentos antigos.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for name in ('credenciais', 'dados_externos', 'modelos', 'resultados'):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT, '| livre reportado:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')

Mounted at /content/drive
Drive: /content/drive/MyDrive/TCC_Final | livre reportado: 5.6 GB
/content livre: 252.95 GB


In [3]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

Credencial desbloqueada nesta sessão.


In [4]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'main'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Mirage-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('experimentos/benchmark_mirage_2000.py', 'Notebooks/fase_3/fase3_6/avaliar_fase3_6_mirage.py', 'experimentos/avaliar_benchmark_2000.py', 'Notebooks/fase_1/fase1_vit.ipynb', 'Notebooks/fase_3/fase3_3/fase3.3_compressao.ipynb')
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc35_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro o avaliador da Fase 3.6 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

Baixando snapshot do GitHub...
Código sincronizado: 83a5fa8c112d522765348f0ebeda66f0b420cb27


In [5]:
# 5. Comandos e progresso periódico; nenhum treino é iniciado aqui.
import os, queue, subprocess, sys, threading, time
BENCH = CODE_ROOT / 'experimentos/benchmark_mirage_2000.py'
EVAL = CODE_ROOT / 'Notebooks/fase_3/fase3_6/avaliar_fase3_6_mirage.py'
for path in (BENCH, EVAL):
    if not path.is_file(): raise FileNotFoundError(path)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)


Código: /content/tcc_codigo/83a5fa8c112d522765348f0ebeda66f0b420cb27


## Etapas manuais

**Copia local:** reconstrui `/content` a partir do Drive e confere hashes; nao refaz a selecao. **Congelamento:** verifica checkpoint, calibracao, manifesto e resultado da 3.5 antes da inferencia. **Avaliacao:** executa a GPU uma vez e grava metricas separadas. Se ja existir resultado, o script recusa sobrescreve-lo.


In [6]:
# 6. Copiar e verificar as mesmas 2.000 imagens Mirage ja congeladas.
run_live('copia local Mirage', [sys.executable, '-u', str(BENCH), 'stage', '--root', str(ROOT)])


copia local Mirage: iniciando
Cópia/verificação: 100/2000
Cópia/verificação: 200/2000
Cópia/verificação: 300/2000
Cópia/verificação: 400/2000
Cópia/verificação: 500/2000
Cópia/verificação: 600/2000
Cópia/verificação: 700/2000
Cópia/verificação: 800/2000
Cópia/verificação: 900/2000
Cópia/verificação: 1000/2000
Cópia/verificação: 1100/2000
Cópia/verificação: 1200/2000
Cópia/verificação: 1300/2000
Cópia/verificação: 1400/2000
Cópia/verificação: 1500/2000
Cópia/verificação: 1600/2000
Cópia/verificação: 1700/2000
Cópia/verificação: 1800/2000
Cópia/verificação: 1900/2000
Cópia/verificação: 2000/2000
Benchmark local pronto: /content/benchmark_externo_mirage_2000_v1
copia local Mirage: concluído em 7.5 min


In [7]:
# 7. Congelar configuracao da 3.6 antes de qualquer inferencia externa.
run_live('congelamento Mirage 3.6', [sys.executable, '-u', str(EVAL), 'freeze', '--root', str(ROOT), '--code-root', str(CODE_ROOT)])


congelamento Mirage 3.6: iniciando
Configuração 3.6 congelada antes da inferência: /content/drive/MyDrive/TCC_Final/resultados/benchmark_externo_mirage_2000_v1/locks/fase3_6_escala_p05_v1.json
{
  "model": "fase3_6_escala_p05_v1",
  "checkpoint_sha256": "5f21f1246f91258dd5e2fe36e3062127587f8b4acc62fadf7b0ad73fb6b2783b",
  "config_sha256": "f40079464699ed9f250787c445b1eaeb1df06281882520256c2ab3430ae96971",
  "internal_metrics_sha256": "d81045baf7ecb1f319b652b78bb813277cd71e0281fb19a5720b88aa5126d7b6",
  "architecture_source_sha256": "3db23c93de1130c4b55a1b1ad2f3f522005e5e2dd3d15733698f1f2418eb7c7b",
  "evaluation_script_sha256": "15a058a4bf50181100b6300a2470e153312faa1c6dd1053484d306fc7cd8b82c",
  "manifest_sha256": "4b3658f880e022957557933fd662c69485c19a5ed9a04e2e09c32ce485737190",
  "prior_metrics_sha256": "0f88e95ed7458059eb8dc09438d64aac0a20cdea399d3a6ff88b16f38d0a057b",
  "prior_predictions_sha256": "ec6a12597b23a82e219c2e417dbb72ee86d5330fb3cbe36815ad368704bbcbb2",
  "temperature"

In [8]:
# 8. AVALIACAO EXTERNA MANUAL. Nao treina nem recalibra.
run_live('avaliacao Mirage 3.6', [sys.executable, '-u', str(EVAL), 'evaluate', '--root', str(ROOT), '--code-root', str(CODE_ROOT)])


avaliacao Mirage 3.6: iniciando
Integridade antes da inferência: 100/2000
Integridade antes da inferência: 200/2000
Integridade antes da inferência: 300/2000
Integridade antes da inferência: 400/2000
Integridade antes da inferência: 500/2000
Integridade antes da inferência: 600/2000
Integridade antes da inferência: 700/2000
Integridade antes da inferência: 800/2000
Integridade antes da inferência: 900/2000
Integridade antes da inferência: 1000/2000
Integridade antes da inferência: 1100/2000
Integridade antes da inferência: 1200/2000
Integridade antes da inferência: 1300/2000
Integridade antes da inferência: 1400/2000
Integridade antes da inferência: 1500/2000
Integridade antes da inferência: 1600/2000
Integridade antes da inferência: 1700/2000
Integridade antes da inferência: 1800/2000
Integridade antes da inferência: 1900/2000
Integridade antes da inferência: 2000/2000
Fase 3.6 congelada | GPU=Tesla T4 | T=0.466226 | limiar=0.495000
avaliacao Mirage 3.6: ativo há 0.7 min; sem nova saí

In [9]:
# 9. Resultados somente leitura e comparacao com a Fase 3.5.
result_dir = ROOT / 'resultados/benchmark_externo_mirage_2000_v1/fase3_6_escala_p05_v1'
previous_dir = ROOT / 'resultados/benchmark_externo_mirage_2000_v1/fase3_5_cfsmall_v1'
for label, directory in [('3.5', previous_dir), ('3.6', result_dir)]:
    path = directory / 'metrics.json'
    if path.is_file():
        metrics = json.loads(path.read_text(encoding='utf-8'))
        print(label, 'accuracy=', metrics['accuracy'], 'AUC=', metrics['auc'], 'fake_recall=', metrics['recall_fake'], 'real_specificity=', metrics['specificity_real'])
        if label == '3.6': print('Paired correct:', metrics['paired_correctness'])
    else:
        print(label, 'avaliacao indisponivel:', path)


3.5 accuracy= 0.6045 AUC= 0.679972 fake_recall= 0.351 real_specificity= 0.858
3.6 accuracy= 0.572 AUC= 0.6303885 fake_recall= 0.288 real_specificity= 0.856
Paired correct: {'both_correct': 1123, 'only_phase3_5_correct': 86, 'only_phase3_6_correct': 21, 'both_wrong': 770}
